# Flight Delay Analysis using PySpark

## Exploratory Data Analysis (EDA)

This notebook combines the original notebooks and organizes them with markdown explanations.

For each section:
- **Objective** – What we are doing
- **Why?** – Purpose of the step
- **ML Benefit** – How it helps build a machine learning model


## 2. Load Dataset
**Objective:** Load the dataset into a Spark DataFrame.

**Why?** Make the raw data available for analysis.

**ML Benefit:** First step before preprocessing and model training.


In [2]:
df = spark.read.parquet(
    "s3://shubham-airline-dataset/Silver/Flight_Data_2020_2025/"
)

## 3. Select Relevant Columns
**Objective:** Keep only required columns.

**Why?** Reduce unnecessary data.

**ML Benefit:** Improves efficiency and avoids irrelevant features.


In [4]:
eda_df = df.select(
    "FlightDate",
    "Year",
    "Quarter",
    "Month",
    "DayOfMonth",
    "DayOfWeek",
    "Marketing_Airline_Network",
    "Origin",
    "OriginState",
    "Dest",
    "DestState",
    "CRSDepTime",
    "CRSArrTime",
    "DepDelay",
    "DepDel15",
    "ArrDelay",
    "ArrDel15",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
    "AirTime",
    "Distance",
    "TaxiOut",
    "TaxiIn",
    "Cancelled",
    "Diverted"
)

In [5]:
print("Rows :", eda_df.count())
print("Columns :", len(eda_df.columns))

('Rows :', 18527443)
('Columns :', 28)

## 5. Preview Dataset
**Objective:** Display sample records.

**ML Benefit:** Validate data before analysis.


In [6]:
eda_df.show(5, truncate=False)

+-------------------+----+-------+-----+----------+---------+-------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+-------+--------+-------+------+---------+--------+
|FlightDate         |Year|Quarter|Month|DayOfMonth|DayOfWeek|Marketing_Airline_Network|Origin|OriginState|Dest|DestState|CRSDepTime|CRSArrTime|DepDelay|DepDel15|ArrDelay|ArrDel15|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|AirTime|Distance|TaxiOut|TaxiIn|Cancelled|Diverted|
+-------------------+----+-------+-----+----------+---------+-------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+-------+--------+-------+------+---------+--------+
|2024-06-09 00:00:00|2024|2      |6    |9         |7        |AA                       |CAE   |SC         |

## 1. Import Required Libraries
**Objective:** Import required PySpark and Python libraries.

**Why?** Access data processing, analysis and visualization functions.

**ML Benefit:** Provides tools for preprocessing and feature engineering.


In [9]:
from pyspark.sql.types import *

numerical_cols = []
categorical_cols = []
date_cols = []

for field in eda_df.schema.fields:
    if isinstance(field.dataType, (IntegerType, LongType, FloatType, DoubleType, DecimalType)):
        numerical_cols.append(field.name)
    elif isinstance(field.dataType, (StringType, BooleanType)):
        categorical_cols.append(field.name)
    elif isinstance(field.dataType, (DateType, TimestampType)):
        date_cols.append(field.name)

print("Numerical Columns:")
print(numerical_cols)

print("\nCategorical Columns:")
print(categorical_cols)

print("\nDate Columns:")
print(date_cols)

Numerical Columns:
['Year', 'Quarter', 'Month', 'DayOfMonth', 'DayOfWeek', 'CRSDepTime', 'CRSArrTime', 'DepDelay', 'DepDel15', 'ArrDelay', 'ArrDel15', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'AirTime', 'Distance', 'TaxiOut', 'TaxiIn', 'Cancelled', 'Diverted']

Categorical Columns:
['Marketing_Airline_Network', 'Origin', 'OriginState', 'Dest', 'DestState']

Date Columns:
['FlightDate']

Delay Reason Columns
CarrierDelay
WeatherDelay
NASDelay
SecurityDelay
LateAircraftDelay

These are only populated when a flight is delayed. If a flight is on time, all these columns remain NULL.

## 6. Missing Value Analysis
**Objective:** Identify missing values.

**ML Benefit:** Decide whether to remove or impute missing data.


In [13]:
from pyspark.sql.functions import col, count, when

total_rows = eda_df.count()

missing = eda_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in eda_df.columns
])

missing.show()

+----------+----+-------+-----+----------+---------+-------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+-------+--------+-------+------+---------+--------+
|FlightDate|Year|Quarter|Month|DayOfMonth|DayOfWeek|Marketing_Airline_Network|Origin|OriginState|Dest|DestState|CRSDepTime|CRSArrTime|DepDelay|DepDel15|ArrDelay|ArrDel15|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|AirTime|Distance|TaxiOut|TaxiIn|Cancelled|Diverted|
+----------+----+-------+-----+----------+---------+-------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+-------+--------+-------+------+---------+--------+
|         0|   0|      0|    0|         0|        0|                        0|     0|          0|   0|        0|         0|         0

In [14]:
from pyspark.sql.functions import col

missing_percent = eda_df.select([
    (
        count(when(col(c).isNull(), c)) / total_rows * 100
    ).alias(c)
    for c in eda_df.columns
])

missing_percent.show(truncate=False)

+----------+----+-------+-----+----------+---------+-------------------------+------+-----------+----+---------+----------+----------+------------------+------------------+------------------+------------------+-----------------+-----------------+-----------------+-----------------+-----------------+------------------+--------+-----------------+------------------+---------+--------+
|FlightDate|Year|Quarter|Month|DayOfMonth|DayOfWeek|Marketing_Airline_Network|Origin|OriginState|Dest|DestState|CRSDepTime|CRSArrTime|DepDelay          |DepDel15          |ArrDelay          |ArrDel15          |CarrierDelay     |WeatherDelay     |NASDelay         |SecurityDelay    |LateAircraftDelay|AirTime           |Distance|TaxiOut          |TaxiIn            |Cancelled|Diverted|
+----------+----+-------+-----+----------+---------+-------------------------+------+-----------+----+---------+----------+----------+------------------+------------------+------------------+------------------+-----------------+--

## 7. Duplicate Analysis
**Objective:** Detect duplicate records.

**ML Benefit:** Prevent model bias from repeated observations.


In [15]:
duplicate_rows = eda_df.count() - eda_df.dropDuplicates().count()

print("Duplicate Rows:", duplicate_rows)

('Duplicate Rows:', 10)

## 9. Categorical Analysis
**Objective:** Analyze frequencies.

**ML Benefit:** Understand categories for encoding.


In [16]:
from pyspark.sql.functions import count

duplicates = (
    eda_df
    .groupBy(eda_df.columns)
    .count()
    .filter("count > 1")
)

duplicates.show(truncate=False)

+-------------------+----+-------+-----+----------+---------+-------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+-------+--------+-------+------+---------+--------+-----+
|FlightDate         |Year|Quarter|Month|DayOfMonth|DayOfWeek|Marketing_Airline_Network|Origin|OriginState|Dest|DestState|CRSDepTime|CRSArrTime|DepDelay|DepDel15|ArrDelay|ArrDel15|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|AirTime|Distance|TaxiOut|TaxiIn|Cancelled|Diverted|count|
+-------------------+----+-------+-----+----------+---------+-------------------------+------+-----------+----+---------+----------+----------+--------+--------+--------+--------+------------+------------+--------+-------------+-----------------+-------+--------+-------+------+---------+--------+-----+
|2020-03-25 00:00:00|2020|1      |3    |25        |3        |DL                       |J

In [17]:
candidate_key = [
    "FlightDate",
    "Marketing_Airline_Network",
    "Origin",
    "Dest",
    "CRSDepTime"
]

total_rows = eda_df.count()
unique_rows = eda_df.select(candidate_key).dropDuplicates().count()

print("Total Rows :", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicates :", total_rows - unique_rows)

('Total Rows :', 18527443)
('Unique Rows:', 18527225)
('Duplicates :', 218)

In [18]:
candidate_key = [
    "FlightDate",
    "Marketing_Airline_Network",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime"
]

total = eda_df.count()
unique = eda_df.select(candidate_key).dropDuplicates().count()

print("Duplicates:", total - unique)


('Duplicates:', 191)

In [22]:
from pyspark.sql.functions import countDistinct

eda_df.select([
    countDistinct(c).alias(c)
    for c in eda_df.columns
]).show(vertical=True, truncate=False)

-RECORD 0-------------------------
 FlightDate                | 2130 
 Year                      | 6    
 Quarter                   | 4    
 Month                     | 12   
 DayOfMonth                | 31   
 DayOfWeek                 | 7    
 Marketing_Airline_Network | 10   
 Origin                    | 387  
 OriginState               | 53   
 Dest                      | 388  
 DestState                 | 53   
 CRSDepTime                | 1408 
 CRSArrTime                | 1439 
 DepDelay                  | 2160 
 DepDel15                  | 2    
 ArrDelay                  | 2183 
 ArrDel15                  | 2    
 CarrierDelay              | 1953 
 WeatherDelay              | 1260 
 NASDelay                  | 1104 
 SecurityDelay             | 281  
 LateAircraftDelay         | 1540 
 AirTime                   | 698  
 Distance                  | 1815 
 TaxiOut                   | 205  
 TaxiIn                    | 275  
 Cancelled                 | 2    
 Diverted           

In [23]:
for col_name, dtype in eda_df.dtypes:
    print("{:<30} {}".format(col_name, dtype))

FlightDate                     timestamp
Year                           int
Quarter                        int
Month                          int
DayOfMonth                     int
DayOfWeek                      int
Marketing_Airline_Network      string
Origin                         string
OriginState                    string
Dest                           string
DestState                      string
CRSDepTime                     int
CRSArrTime                     int
DepDelay                       double
DepDel15                       double
ArrDelay                       double
ArrDel15                       double
CarrierDelay                   double
WeatherDelay                   double
NASDelay                       double
SecurityDelay                  double
LateAircraftDelay              double
AirTime                        double
Distance                       double
TaxiOut                        double
TaxiIn                         double
Cancelled                     

In [24]:
from pyspark.sql import Row

schema_df = spark.createDataFrame(
    [Row(Column=col_name, DataType=dtype) for col_name, dtype in eda_df.dtypes]
)

schema_df.show(truncate=False)

----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 37496)
----------------------------------------
+-------------------------+---------+
|Column                   |DataType |
+-------------------------+---------+
|FlightDate               |timestamp|
|Year                     |int      |
|Quarter                  |int      |
|Month                    |int      |
|DayOfMonth               |int      |
|DayOfWeek                |int      |
|Marketing_Airline_Network|string   |
|Origin                   |string   |
|OriginState              |string   |
|Dest                     |string   |
|DestState                |string   |
|CRSDepTime               |int      |
|CRSArrTime               |int      |
|DepDelay                 |double   |
|DepDel15                 |double   |
|ArrDelay                 |double   |
|ArrDel15                 |double   |
|CarrierDelay             |double   |
|WeatherDelay             |double   |
|NASDela

In [25]:
for column in eda_df.columns:
    print("=" * 60)
    print("Column:", column)
    eda_df.select(column).distinct().show(5, truncate=False)

('Column:', 'FlightDate')
+-------------------+
|FlightDate         |
+-------------------+
|2023-04-04 00:00:00|
|2021-08-27 00:00:00|
|2025-09-23 00:00:00|
|2024-09-22 00:00:00|
|2024-09-27 00:00:00|
+-------------------+
only showing top 5 rows

('Column:', 'Year')
+----+
|Year|
+----+
|2025|
|2023|
|2022|
|2020|
|2024|
+----+
only showing top 5 rows

('Column:', 'Quarter')
+-------+
|Quarter|
+-------+
|1      |
|3      |
|4      |
|2      |
+-------+

('Column:', 'Month')
+-----+
|Month|
+-----+
|12   |
|1    |
|6    |
|3    |
|5    |
+-----+
only showing top 5 rows

('Column:', 'DayOfMonth')
+----------+
|DayOfMonth|
+----------+
|31        |
|28        |
|26        |
|27        |
|12        |
+----------+
only showing top 5 rows

('Column:', 'DayOfWeek')
+---------+
|DayOfWeek|
+---------+
|1        |
|6        |
|3        |
|5        |
|4        |
+---------+
only showing top 5 rows

('Column:', 'Marketing_Airline_Network')
+-------------------------+
|Marketing_Airline_Network

## 8. Summary Statistics
**Objective:** Compute descriptive statistics.

**ML Benefit:** Understand distributions before feature engineering.


In [27]:
eda_df.describe().show(truncate=False)

+-------+------------------+------------------+------------------+------------------+-----------------+-------------------------+--------+-----------+--------+---------+-----------------+------------------+------------------+-------------------+-----------------+-------------------+------------------+-----------------+------------------+-------------------+------------------+------------------+-----------------+------------------+------------------+-------------------+--------------------+
|summary|Year              |Quarter           |Month             |DayOfMonth        |DayOfWeek        |Marketing_Airline_Network|Origin  |OriginState|Dest    |DestState|CRSDepTime       |CRSArrTime        |DepDelay          |DepDel15           |ArrDelay         |ArrDel15           |CarrierDelay      |WeatherDelay     |NASDelay          |SecurityDelay      |LateAircraftDelay |AirTime           |Distance         |TaxiOut           |TaxiIn            |Cancelled          |Diverted            |
+-------+-

In [28]:
numeric_cols = [
    "DepDelay",
    "ArrDelay",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
    "AirTime",
    "Distance",
    "TaxiOut",
    "TaxiIn"
]

eda_df.select(numeric_cols).summary(
    "count",
    "mean",
    "stddev",
    "min",
    "25%",
    "50%",
    "75%",
    "max"
).show(truncate=False)

+-------+------------------+-----------------+------------------+-----------------+------------------+-------------------+------------------+------------------+-----------------+------------------+------------------+
|summary|DepDelay          |ArrDelay         |CarrierDelay      |WeatherDelay     |NASDelay          |SecurityDelay      |LateAircraftDelay |AirTime           |Distance         |TaxiOut           |TaxiIn            |
+-------+------------------+-----------------+------------------+-----------------+------------------+-------------------+------------------+------------------+-----------------+------------------+------------------+
|count  |18164368          |18111025         |3459828           |3459828          |3459828           |3459828            |3459828           |18111025          |18527443         |18156387          |18151299          |
|mean   |10.636345068542985|4.890034329917826|23.529106938263983|3.66678950514303 |13.24276784857513 |0.13767418495948353|27.8211896

In [29]:
import matplotlib.pyplot as plt

top_airlines = (
    eda_df.groupBy("Marketing_Airline_Network")
          .count()
          .orderBy("count", ascending=False)
          .toPandas()
)

plt.figure(figsize=(10,5))
plt.bar(top_airlines["Marketing_Airline_Network"], top_airlines["count"])
plt.xticks(rotation=45)
plt.title("Flights by Airline")
plt.xlabel("Airline")
plt.ylabel("Number of Flights")
plt.show()

No module named matplotlib.pyplot
Traceback (most recent call last):
ImportError: No module named matplotlib.pyplot



In [30]:
numeric_cols = [
    "DepDelay",
    "ArrDelay",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
    "AirTime",
    "Distance",
    "TaxiOut",
    "TaxiIn"
]

## 10. Numerical Distribution
**Objective:** Visualize numerical distributions.

**ML Benefit:** Detect skewness and outliers.


In [31]:
import matplotlib.pyplot as plt

pdf = eda_df.select("DepDelay").dropna().toPandas()

plt.figure(figsize=(8,5))
plt.hist(pdf["DepDelay"], bins=50)
plt.title("Departure Delay Distribution")
plt.xlabel("Minutes")
plt.ylabel("Frequency")
plt.show()

No module named matplotlib.pyplot
Traceback (most recent call last):
ImportError: No module named matplotlib.pyplot



## 12. Skewness Analysis
**Objective:** Measure asymmetry.

**ML Benefit:** Determine transformations.


In [32]:
from pyspark.sql.functions import skewness

for col in numeric_cols:
    print(col)
    eda_df.select(skewness(col)).show()

DepDelay
+------------------+
|skewness(DepDelay)|
+------------------+
|12.230282371838284|
+------------------+

ArrDelay
+------------------+
|skewness(ArrDelay)|
+------------------+
|11.022368646036275|
+------------------+

CarrierDelay
+----------------------+
|skewness(CarrierDelay)|
+----------------------+
|    11.557526255969059|
+----------------------+

WeatherDelay
+----------------------+
|skewness(WeatherDelay)|
+----------------------+
|    20.218917872212042|
+----------------------+

NASDelay
+------------------+
|skewness(NASDelay)|
+------------------+
|10.706932399660138|
+------------------+

SecurityDelay
+-----------------------+
|skewness(SecurityDelay)|
+-----------------------+
|      96.19388725213349|
+-----------------------+

LateAircraftDelay
+---------------------------+
|skewness(LateAircraftDelay)|
+---------------------------+
|         7.5631343170314125|
+---------------------------+

AirTime
+------------------+
| skewness(AirTime)|
+------------

# Additional EDA Analysis
The following sections continue the exploratory analysis from the second notebook.

In [1]:
spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1783610551559_0001,pyspark3,idle,Link,Link,✔


SparkSession available as 'spark'.

In [2]:
df = spark.read.parquet(
    "s3://shubham-airline-dataset/Silver/Flight_Data_2020_2025/"
)

## 4. Inspect Dataset Schema
**Objective:** Verify column names and data types.

**ML Benefit:** Helps identify categorical and numerical features.


In [3]:
df.printSchema()

root
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: timestamp (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Operated_or_Branded_Code_Share_Partners: string (nullable = true)
 |-- DOT_ID_Marketing_Airline: integer (nullable = true)
 |-- IATA_Code_Marketing_Airline: string (nullable = true)
 |-- Flight_Number_Marketing_Airline: integer (nullable = true)
 |-- Originally_Scheduled_Code_Share_Airline: string (nullable = true)
 |-- DOT_ID_Originally_Scheduled_Code_Share_Airline: integer (nullable = true)
 |-- IATA_Code_Originally_Scheduled_Code_Share_Airline: string (nullable = true)
 |-- Flight_Num_Originally_Scheduled_Code_Share_Airline: integer (nullable = true)
 |-- Operating_Airline: string (nullable = true)
 |-- DOT_ID_Operating_Airline: integer (nullable = true)
 |-- IATA_Code_Operating_Airline: string (nullable = true)


In [4]:
df.show(5, truncate=False)

+-------+-----+----------+---------+-------------------+-------------------------+---------------------------------------+------------------------+---------------------------+-------------------------------+---------------------------------------+----------------------------------------------+-------------------------------------------------+--------------------------------------------------+-----------------+------------------------+---------------------------+-----------+-------------------------------+---------------+------------------+------------------+------+--------------+-----------+---------------+---------------+---------+-------------+----------------+----------------+----+-------------+---------+-------------+--------------+-------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+----------------+--------+--------

In [5]:
eda_df = df.select(
    "FlightDate",
    "AirTime",
    "Cancelled",
    "Diverted",
    "Distance",
    "TaxiOut",
    "TaxiIn"
)

In [6]:
eda_df.show(5, truncate=False)

+-------------------+-------+---------+--------+--------+-------+------+
|FlightDate         |AirTime|Cancelled|Diverted|Distance|TaxiOut|TaxiIn|
+-------------------+-------+---------+--------+--------+-------+------+
|2024-06-09 00:00:00|33.0   |0.0      |0.0     |88.0    |51.0   |19.0  |
|2024-06-10 00:00:00|26.0   |0.0      |0.0     |88.0    |20.0   |18.0  |
|2024-06-11 00:00:00|26.0   |0.0      |0.0     |88.0    |14.0   |16.0  |
|2024-06-12 00:00:00|33.0   |0.0      |0.0     |88.0    |39.0   |10.0  |
|2024-06-13 00:00:00|30.0   |0.0      |0.0     |88.0    |23.0   |27.0  |
+-------------------+-------+---------+--------+--------+-------+------+
only showing top 5 rows

In [7]:
print("Number of Columns :", len(eda_df.columns))

Number of Columns : 7

In [8]:
print("Column Names")

for col in eda_df.columns:
    print(col)

Column Names
FlightDate
AirTime
Cancelled
Diverted
Distance
TaxiOut
TaxiIn

In [9]:
numerical_columns = [
    "AirTime",
    "Distance",
    "TaxiOut",
    "TaxiIn"
]

categorical_columns = [
    "Cancelled",
    "Diverted"
]

datetime_columns = [
    "FlightDate"
]

print("Numerical :", numerical_columns)
print("Categorical :", categorical_columns)
print("Datetime :", datetime_columns)

Numerical : ['AirTime', 'Distance', 'TaxiOut', 'TaxiIn']
Categorical : ['Cancelled', 'Diverted']
Datetime : ['FlightDate']

# Column Classification

In [11]:
for column_name, datatype in eda_df.dtypes:
    print(f"{column_name:<15} : {datatype}")

FlightDate      : timestamp
AirTime         : double
Cancelled       : double
Diverted        : double
Distance        : double
TaxiOut         : double
TaxiIn          : double

In [10]:
numerical_columns = [
    "AirTime",
    "Distance",
    "TaxiOut",
    "TaxiIn"
]

categorical_columns = [
    "Cancelled",
    "Diverted"
]

datetime_columns = [
    "FlightDate"
]

print("Numerical :", numerical_columns)
print("Categorical :", categorical_columns)
print("Datetime :", datetime_columns)

Numerical : ['AirTime', 'Distance', 'TaxiOut', 'TaxiIn']
Categorical : ['Cancelled', 'Diverted']
Datetime : ['FlightDate']

# Format Consistency (check for null timstamps)

In [13]:
eda_df.filter(col("FlightDate").isNull()).count()

'str' object is not callable
Traceback (most recent call last):
TypeError: 'str' object is not callable



In [14]:
print(col)

TaxiIn

In [15]:
from pyspark.sql.functions import col

In [16]:
eda_df.filter(col("FlightDate").isNull()).count()

0

### Observation

The `FlightDate` column contains **0 null values**, indicating that every flight record has a valid timestamp. Therefore, no missing value treatment is required for this column, and it is suitable for time-series analysis.

Validate(canclled column contain only valid binary values

In [17]:
eda_df.groupBy("Cancelled") \
      .count() \
      .orderBy("Cancelled") \
      .show()

+---------+--------+
|Cancelled|   count|
+---------+--------+
|      0.0|18154858|
|      1.0|  372585|
+---------+--------+

0 → Flight was not cancelled
1 → Flight was cancelled

valid Diverted

In [18]:
eda_df.groupBy("Diverted") \
      .count() \
      .orderBy("Diverted") \
      .show()

+--------+--------+
|Diverted|   count|
+--------+--------+
|     0.0|18483612|
|     1.0|   43831|
+--------+--------+

Check for Invalid AirTime

In [19]:
eda_df.filter(col("AirTime") <= 0).count()

0

Check for Invalid Distance

In [20]:
eda_df.filter(col("Distance") <= 0).count()

0

Check for Negative TaxiOut

In [21]:
eda_df.filter(col("TaxiOut") < 0).count()

0

In [22]:
eda_df.filter(col("TaxiIn") < 0).count()

0

In [23]:
eda_df.groupBy("Cancelled") \
      .count() \
      .orderBy("Cancelled") \
      .show()

+---------+--------+
|Cancelled|   count|
+---------+--------+
|      0.0|18154858|
|      1.0|  372585|
+---------+--------+

| Value | Meaning              |
| ----- | -------------------- |
| 0     | Flight was completed |
| 1     | Flight was cancelled |


In [24]:
eda_df.groupBy("Diverted") \
      .count() \
      .orderBy("Diverted") \
      .show()

+--------+--------+
|Diverted|   count|
+--------+--------+
|     0.0|18483612|
|     1.0|   43831|
+--------+--------+

| Value | Meaning                           |
| ----- | --------------------------------- |
| 0     | Flight followed the planned route |
| 1     | Flight was diverted               |


In [25]:
eda_df.describe().show()

+-------+------------------+-------------------+--------------------+-----------------+------------------+------------------+
|summary|           AirTime|          Cancelled|            Diverted|         Distance|           TaxiOut|            TaxiIn|
+-------+------------------+-------------------+--------------------+-----------------+------------------+------------------+
|  count|          18111025|           18527443|            18527443|         18527443|          18156387|          18151299|
|   mean|111.48739135416135|0.02010989859744812|0.002365733900787065|802.7952166955796|16.965521224018854|7.9830048527105415|
| stddev| 68.90486886843985|0.14037625026845715|  0.0485812446452723|581.9854839475836| 9.414446706480527|6.5418253437790925|
|    min|               6.0|                0.0|                 0.0|             21.0|               0.0|               0.0|
|    max|             953.0|                1.0|                 1.0|           5095.0|            1162.0|            

The counts are not equal across all columns.
some columns contain missing values.

In [26]:
from pyspark.sql.functions import count, when, col

eda_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in eda_df.columns
]).show()

+----------+-------+---------+--------+--------+-------+------+
|FlightDate|AirTime|Cancelled|Diverted|Distance|TaxiOut|TaxiIn|
+----------+-------+---------+--------+--------+-------+------+
|         0| 416418|        0|       0|       0| 371056|376144|
+----------+-------+---------+--------+--------+-------+------+

Null Values For Every Column

In [27]:
total_rows = eda_df.count()

eda_df.select([
    (
        count(when(col(c).isNull(), c)) / total_rows * 100
    ).alias(c)
    for c in eda_df.columns
]).show()

+----------+------------------+---------+--------+--------+-----------------+------------------+
|FlightDate|           AirTime|Cancelled|Diverted|Distance|          TaxiOut|            TaxiIn|
+----------+------------------+---------+--------+--------+-----------------+------------------+
|       0.0|2.2475740446212678|      0.0|     0.0|     0.0|2.002737236865335|2.0301992023400097|
+----------+------------------+---------+--------+--------+-----------------+------------------+

Null Percentage

In [28]:
total_rows = eda_df.count()

eda_df.select([
    (
        count(when(col(c).isNull(), c)) / total_rows * 100
    ).alias(c)
    for c in eda_df.columns
]).show()

+----------+------------------+---------+--------+--------+-----------------+------------------+
|FlightDate|           AirTime|Cancelled|Diverted|Distance|          TaxiOut|            TaxiIn|
+----------+------------------+---------+--------+--------+-----------------+------------------+
|       0.0|2.2475740446212678|      0.0|     0.0|     0.0|2.002737236865335|2.0301992023400097|
+----------+------------------+---------+--------+--------+-----------------+------------------+

Count Distinct Values(check uniqueness of column s)

In [29]:
from pyspark.sql.functions import countDistinct

for column in eda_df.columns:
    distinct = eda_df.select(countDistinct(column)).first()[0]
    print(f"{column} : {distinct}")

FlightDate : 2130
AirTime : 698
Cancelled : 2
Diverted : 2
Distance : 1815
TaxiOut : 205
TaxiIn : 275

Duplicate values (column wise)

In [30]:
from pyspark.sql.functions import countDistinct

total = eda_df.count()

for column in eda_df.columns:
    distinct = eda_df.select(countDistinct(column)).first()[0]

    print("=" * 50)
    print("Column :", column)
    print("Total Values :", total)
    print("Distinct Values :", distinct)
    print("Duplicate Values :", total - distinct)

Column : FlightDate
Total Values : 18527443
Distinct Values : 2130
Duplicate Values : 18525313
Column : AirTime
Total Values : 18527443
Distinct Values : 698
Duplicate Values : 18526745
Column : Cancelled
Total Values : 18527443
Distinct Values : 2
Duplicate Values : 18527441
Column : Diverted
Total Values : 18527443
Distinct Values : 2
Duplicate Values : 18527441
Column : Distance
Total Values : 18527443
Distinct Values : 1815
Duplicate Values : 18525628
Column : TaxiOut
Total Values : 18527443
Distinct Values : 205
Duplicate Values : 18527238
Column : TaxiIn
Total Values : 18527443
Distinct Values : 275
Duplicate Values : 18527168

Null Percentage

In [31]:
total_rows = eda_df.count()

null_percentage = eda_df.select([
    (
        count(when(col(c).isNull(), c)) / total_rows * 100
    ).alias(c)
    for c in eda_df.columns
])

null_percentage.show()

+----------+------------------+---------+--------+--------+-----------------+------------------+
|FlightDate|           AirTime|Cancelled|Diverted|Distance|          TaxiOut|            TaxiIn|
+----------+------------------+---------+--------+--------+-----------------+------------------+
|       0.0|2.2475740446212678|      0.0|     0.0|     0.0|2.002737236865335|2.0301992023400097|
+----------+------------------+---------+--------+--------+-----------------+------------------+

In [32]:
from pyspark.sql.functions import col, count, when

null_count = eda_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in eda_df.columns
])

null_count.show()

+----------+-------+---------+--------+--------+-------+------+
|FlightDate|AirTime|Cancelled|Diverted|Distance|TaxiOut|TaxiIn|
+----------+-------+---------+--------+--------+-------+------+
|         0| 416418|        0|       0|       0| 371056|376144|
+----------+-------+---------+--------+--------+-------+------+

In [33]:
eda_df.filter(
    col("AirTime").isNull()
).groupBy("Cancelled").count().show()

+---------+------+
|Cancelled| count|
+---------+------+
|      0.0| 43833|
|      1.0|372585|
+---------+------+

Almost 90% of missing AirTime values belong to cancelled flights.

In [34]:
eda_df.filter(
    col("TaxiOut").isNull()
).groupBy("Cancelled").count().show()

+---------+------+
|Cancelled| count|
+---------+------+
|      1.0|371056|
+---------+------+

In [35]:
eda_df.filter(
    col("TaxiIn").isNull()
).groupBy("Cancelled").count().show()

+---------+------+
|Cancelled| count|
+---------+------+
|      0.0|  3559|
|      1.0|372585|
+---------+------+

Almost 99% of missing TaxiIn values occur because the flight was cancelled.

# Duplicate Records Check

In [36]:
total_records = eda_df.count()

distinct_records = eda_df.distinct().count()

duplicate_records = total_records - distinct_records

print("Duplicate Records:", duplicate_records)

Duplicate Records: 296449

In [37]:
duplicate_percentage = (296449 / 18527443) * 100
print(f"Duplicate Percentage: {duplicate_percentage:.4f}%")

Duplicate Percentage: 1.6001%

In [38]:
duplicate_df = (
    eda_df.groupBy(eda_df.columns)
          .count()
          .filter(col("count") > 1)
)

duplicate_df.show(20, truncate=False)

+-------------------+-------+---------+--------+--------+-------+------+-----+
|FlightDate         |AirTime|Cancelled|Diverted|Distance|TaxiOut|TaxiIn|count|
+-------------------+-------+---------+--------+--------+-------+------+-----+
|2024-06-21 00:00:00|113.0  |0.0      |0.0     |834.0   |16.0   |8.0   |2    |
|2024-06-26 00:00:00|51.0   |0.0      |0.0     |279.0   |10.0   |3.0   |2    |
|2024-06-13 00:00:00|null   |1.0      |0.0     |1197.0  |null   |null  |3    |
|2024-06-05 00:00:00|null   |1.0      |0.0     |936.0   |null   |null  |2    |
|2024-06-17 00:00:00|59.0   |0.0      |0.0     |404.0   |11.0   |7.0   |2    |
|2024-06-29 00:00:00|90.0   |0.0      |0.0     |621.0   |13.0   |6.0   |2    |
|2024-06-09 00:00:00|116.0  |0.0      |0.0     |919.0   |10.0   |5.0   |2    |
|2024-06-14 00:00:00|83.0   |0.0      |0.0     |512.0   |12.0   |4.0   |2    |
|2024-06-13 00:00:00|null   |1.0      |0.0     |793.0   |null   |null  |2    |
|2024-06-13 00:00:00|null   |1.0      |0.0     |883.

# Check Flight date format

In [39]:
eda_df.select("FlightDate").show(10, False)

+-------------------+
|FlightDate         |
+-------------------+
|2024-06-09 00:00:00|
|2024-06-10 00:00:00|
|2024-06-11 00:00:00|
|2024-06-12 00:00:00|
|2024-06-13 00:00:00|
|2024-06-14 00:00:00|
|2024-06-15 00:00:00|
|2024-06-16 00:00:00|
|2024-06-17 00:00:00|
|2024-06-18 00:00:00|
+-------------------+
only showing top 10 rows

In [40]:
eda_df.filter(col("AirTime") <= 0).show()

+----------+-------+---------+--------+--------+-------+------+
|FlightDate|AirTime|Cancelled|Diverted|Distance|TaxiOut|TaxiIn|
+----------+-------+---------+--------+--------+-------+------+
+----------+-------+---------+--------+--------+-------+------+

In [41]:
eda_df.filter(col("Distance") <= 0).show()

+----------+-------+---------+--------+--------+-------+------+
|FlightDate|AirTime|Cancelled|Diverted|Distance|TaxiOut|TaxiIn|
+----------+-------+---------+--------+--------+-------+------+
+----------+-------+---------+--------+--------+-------+------+

In [42]:
eda_df.filter(col("TaxiOut") < 0).show()

+----------+-------+---------+--------+--------+-------+------+
|FlightDate|AirTime|Cancelled|Diverted|Distance|TaxiOut|TaxiIn|
+----------+-------+---------+--------+--------+-------+------+
+----------+-------+---------+--------+--------+-------+------+

In [43]:
eda_df.filter(col("TaxiIn") < 0).show()

+----------+-------+---------+--------+--------+-------+------+
|FlightDate|AirTime|Cancelled|Diverted|Distance|TaxiOut|TaxiIn|
+----------+-------+---------+--------+--------+-------+------+
+----------+-------+---------+--------+--------+-------+------+

In [ ]:
print(df.columns)


In [44]:
from pyspark.sql.functions import avg

eda_df.select(
    avg("AirTime").alias("Average_AirTime")
).show()

+------------------+
|   Average_AirTime|
+------------------+
|111.48739135416135|
+------------------+

In [45]:
q1,q2,q3 = eda_df.approxQuantile(
    "AirTime",
    [0.25,0.50,0.75],
    0.01
)

iqr = q3-q1

print("Q1 :",q1)
print("Median :",q2)
print("Q3 :",q3)
print("IQR :",iqr)

Q1 : 63.0
Median : 96.0
Q3 : 143.0
IQR : 80.0

25% of flights have an AirTime of 63 minutes or less.
50% of flights have an AirTime less than or equal to 96 minutes, and 50% have an AirTime greater than 96 minutes.
75% of flights have an AirTime of 143 minutes or less.

In [46]:
lower = q1 - 1.5*iqr
upper = q3 + 1.5*iqr

print(lower)
print(upper)

-57.0
263.0

In [47]:
from pyspark.sql.functions import col

eda_df.filter(
    (col("AirTime") < lower) |
    (col("AirTime") > upper)
).count()

881175

In [48]:
from pyspark.sql.functions import skewness

eda_df.select(
    skewness("AirTime")
).show()

+------------------+
| skewness(AirTime)|
+------------------+
|1.4400037951510338|
+------------------+

In [ ]:
right skewd(Most flights have short to medium air times.)

In [49]:
lower_bound = q1 - (1.5 * iqr)
upper_bound = q3 + (1.5 * iqr)

print("Lower Bound :", lower_bound)
print("Upper Bound :", upper_bound)

Lower Bound : -57.0
Upper Bound : 263.0

Since AirTime cannot be negative, the lower bound of -57 is not meaningful for this dataset.
Any flight with an AirTime greater than 263 minutes will be flagged as a potential outlier by the IQR method.

In [50]:
from pyspark.sql.functions import col

outlier_df = eda_df.filter(col("AirTime") > upper_bound)

outlier_count = outlier_df.count()

print("Number of Outliers :", outlier_count)

Number of Outliers : 881175

Any flight longer than 4 hours 23 minutes is classified as an outlier by the IQR rule.

# Are these outliers actually long-distance flights

In [51]:
outlier_df.select(
    "AirTime",
    "Distance"
).orderBy(col("AirTime").desc()).show(20, truncate=False)

+-------+--------+
|AirTime|Distance|
+-------+--------+
|953.0  |437.0   |
|946.0  |519.0   |
|724.0  |5095.0  |
|723.0  |255.0   |
|715.0  |5095.0  |
|703.0  |5095.0  |
|701.0  |4983.0  |
|699.0  |4983.0  |
|698.0  |4983.0  |
|697.0  |4983.0  |
|695.0  |5095.0  |
|695.0  |4983.0  |
|694.0  |4983.0  |
|694.0  |5095.0  |
|694.0  |5095.0  |
|693.0  |4983.0  |
|692.0  |4983.0  |
|691.0  |5095.0  |
|690.0  |5095.0  |
|690.0  |4983.0  |
+-------+--------+
only showing top 20 rows

In [52]:
from pyspark.sql.functions import when, avg

distance_analysis = eda_df.withColumn(
    "Distance_Category",
    when(col("Distance") < 500, "Short Haul")
    .when((col("Distance") >= 500) & (col("Distance") < 1500), "Medium Haul")
    .otherwise("Long Haul")
)

distance_analysis.groupBy("Distance_Category") \
    .agg(
        avg("AirTime").alias("Average_AirTime"),
        avg("Distance").alias("Average_Distance")
    ) \
    .show(truncate=False)

+-----------------+------------------+------------------+
|Distance_Category|Average_AirTime   |Average_Distance  |
+-----------------+------------------+------------------+
|Short Haul       |52.44538186669035 |303.53214521839016|
|Medium Haul      |120.2124738823188 |871.7318857030558 |
|Long Haul        |256.29616183573177|2055.5379962708207|
+-----------------+------------------+------------------+

| Distance Category | Avg Distance | Avg AirTime |
| ----------------- | -----------: | ----------: |
| Short Haul        |         ~250 |     ~50 min |
| Medium Haul       |         ~900 |    ~120 min |
| Long Haul         |        ~2200 |    ~280 min |


In [53]:
eda_df.select("Distance").describe().show()

+-------+-----------------+
|summary|         Distance|
+-------+-----------------+
|  count|         18527443|
|   mean|802.7952166955796|
| stddev|581.9854839475836|
|    min|             21.0|
|    max|           5095.0|
+-------+-----------------+

# Kpis



Total Flights
Average AirTime
Average Distance
Cancellation Rate (%)
Diversion Rate (%)
Average Taxi-Out Time
Average Taxi-In Time
Flights per Day
Longest Flight
Shortest Flight
Average Daily Flights
Monthly Flight Trend
Yearly Flight Trend

# Features Can Be Create

| Feature          | Formula                                                                                                 |
| ---------------- | ------------------------------------------------------------------------------------------------------- |
| FlightYear       | `year(FlightDate)`                                                                                      |
| FlightMonth      | `month(FlightDate)`                                                                                     |
| Quarter          | `quarter(FlightDate)`                                                                                   |
| DayOfMonth       | `dayofmonth(FlightDate)`                                                                                |
| DayOfWeek        | `dayofweek(FlightDate)`                                                                                 |
| WeekOfYear       | `weekofyear(FlightDate)`                                                                                |
| MonthName        | `date_format(FlightDate,'MMMM')`                                                                        |
| DayName          | `date_format(FlightDate,'EEEE')`                                                                        |
| IsWeekend        | `DayOfWeek IN (1,7)` *(adjust depending on Spark's day numbering)*                                      |
| AirTimeHours     | `AirTime / 60`                                                                                          |
| AirTimeCategory  | `CASE WHEN AirTime <90 THEN 'Short' WHEN AirTime <180 THEN 'Medium' ELSE 'Long' END`                    |
| DistanceKM       | `Distance * 1.60934`                                                                                    |
| DistanceCategory | `CASE WHEN Distance <500 THEN 'Short Haul' WHEN Distance <1500 THEN 'Medium Haul' ELSE 'Long Haul' END` |
| GroundTime       | `TaxiOut + TaxiIn`                                                                                      |
| TotalFlightTime  | `AirTime + TaxiOut + TaxiIn`                                                                            |
| EstimatedSpeed   | `Distance / (AirTime / 60)` *(miles/hour)*                                                              |
| FlightStatus     | `CASE WHEN Cancelled=1 THEN 'Cancelled' WHEN Diverted=1 THEN 'Diverted' ELSE 'Completed' END`           |
| LongFlightFlag   | `AirTime >180`                                                                                          |
| LongHaulFlag     | `Distance >1500`                                                                                        |
| TaxiDelay        | `GroundTime >30`                                                                                        |
